# 07 - İhlal Tespit Değerlendirmesi

Uçtan uca ihlal tespiti doğruluğunun değerlendirilmesi.

## Metrikler
- True Positive, False Positive, False Negative
- Precision, Recall, F1-Score
- Confusion matrix
- Video bazlı sonuçlar

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# Ground truth format (JSON)
# {
#   "video": "test_01.mp4",
#   "violations": [
#     {"frame_start": 100, "frame_end": 150, "vehicle_class": "car", "plate": "34AB123"},
#     ...
#   ]
# }

GT_DIR = Path('data/ground_truth')
RESULTS_DIR = Path('results')

In [ ]:
def load_ground_truth(gt_path):
    """Ground truth JSON dosyasını yükle."""
    with open(gt_path) as f:
        return json.load(f)

def match_violations(gt_violations, pred_violations, frame_tolerance=15):
    """Ground truth ve tahminleri eşleştir."""
    tp, fp, fn = 0, 0, 0
    matched_gt = set()
    
    for pred in pred_violations:
        pred_frame = pred['frame_number']
        found = False
        for i, gt in enumerate(gt_violations):
            if i in matched_gt:
                continue
            if gt['frame_start'] - frame_tolerance <= pred_frame <= gt['frame_end'] + frame_tolerance:
                tp += 1
                matched_gt.add(i)
                found = True
                break
        if not found:
            fp += 1
    
    fn = len(gt_violations) - len(matched_gt)
    return tp, fp, fn

def calculate_metrics(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return {'precision': precision, 'recall': recall, 'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn}

In [ ]:
# Tüm test videoları için değerlendirme
all_results = []

for gt_file in sorted(GT_DIR.glob('*.json')):
    gt_data = load_ground_truth(gt_file)
    video_name = gt_data.get('video', gt_file.stem)
    
    # Pipeline sonuçlarını yükle
    from src.storage.database import ViolationDatabase
    db = ViolationDatabase(str(RESULTS_DIR / 'violations.db'))
    pred_violations = db.get_all_violations(limit=1000)
    db.close()
    
    tp, fp, fn = match_violations(gt_data['violations'], pred_violations)
    metrics = calculate_metrics(tp, fp, fn)
    metrics['video'] = video_name
    all_results.append(metrics)
    
    print(f'{video_name}: P={metrics["precision"]:.3f} R={metrics["recall"]:.3f} F1={metrics["f1"]:.3f}')

if all_results:
    df = pd.DataFrame(all_results)
    print('\nGenel Sonuçlar:')
    print(df.to_string(index=False))
    df.to_csv('results/violation_eval_results.csv', index=False)